# Directed Results: 
# **Core Analyses of Vitamin D Signatures**

This notebook presents the **directed, hypothesis-driven analyses** of transcriptomic responses to Vitamin D and its analogs.  
Unlike exploratory analyses, here we focus on **predefined questions** (core signatures, dose–response, enrichment) using the modular utilities developed in `vitd_utils`.

All constants, parameters, and paths are centralized in `vitd_utils.config`, ensuring reproducibility and consistency across analyses.

## Section 1: Imports & Config.

In [ ]:
# Allow imports from src/vitd_utils
import sys
sys.path.append("../src")

# Core project utilities
from vitd_utils import config, idsymbols, coregenes, dose, gsea, plotting, stats, dataset

# Standard scientific libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display options
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 120)

print("Results will be saved to:", config.RESULTS_DIR)
print("Figures will be saved to:", config.FIG_DIR, "| SAVE_FIGS =", config.SAVE_FIGS)

### 1.1. Load and validate

In [ ]:
import pandas as pd

EXP  = pd.read_parquet("../data/exports/expression_matrix_clean.parquet")
META = pd.read_csv("../data/exports/signature_metadata_clean.csv")

# Estandariza columnas (sig_id, cell_id, dose, analog)
META = dataset.standardize_meta(META)
print("Columns after standardize_meta:", META.columns.tolist())
print(META.filter(regex="dose", axis=1).head(2))  # ver 'dose' presente

# Alinea por sig_id
from vitd_utils import dataset as _ds
EXP, META = _ds.align_exp_meta(EXP, META)


## 2. Gene ID ↔ Symbol Mapping

Most LINCS L1000 resources use **gene IDs** as stable identifiers, while interpretation requires **gene symbols**.  
To ensure consistency, we build a robust mapping between IDs and symbols using `vitd_utils.idsymbols`.  
This step guarantees that downstream analyses (core genes, enrichment, plotting) always have readable gene names with safe fallbacks.


In [ ]:
# Load gene metadata (example: geneinfo_beta.txt already loaded in previous steps)
gene_info = pd.read_csv("../data/raw_data/geneinfo_beta.txt", sep="\t")

# Build ID → symbol mapping
sym_map = idsymbols.build_symbol_map(gene_info)

# Quick check
print("Mapping size:", sym_map.shape[0])
print("Examples:\n", sym_map.head())

# Test safe fallback (ID not in map should return itself)
print("Test mapping:", idsymbols.map_symbols_or_ids(["100", "102", "999"], sym_map)[:5])

## 3. Consensus Core Genes (definition & scoring)

**Goal.** Define a robust Vitamin D “core” signature (UP/DOWN genes) that recurs across contexts (here: cell lines), and compute a single **core score** per signature capturing the balance of UP vs DOWN core genes.

**Method.**
1) Build a gene × context matrix of effects (here: mean L1000 z-scores **per cell line**).
2) For each context, take the top/bottom `N_TOP` genes and **vote-count** across contexts.
3) Select `CORE_UP_N` / `CORE_DN_N` genes using a minimum vote threshold and deterministic tie-breakers (mean |effect|).
4) Compute **core_score** for every signature:  
   `core_score = mean(z(core_UP)) − mean(z(core_DN))` (column-wise centering).

All thresholds and sizes are centralized in `vitd_utils.config`.

In [ ]:
# Sanity alignment: keep only signatures present in both matrices
common_sig = [c for c in EXP.columns if c in set(META["sig_id"])]
EXP = EXP[common_sig].copy()
META = META.loc[META["sig_id"].isin(common_sig)].copy()

# --- Build gene × cell effects (mean across signatures within each cell line)
cell_indexer = META.set_index("sig_id")["cell_id"]
effects_by_cell = EXP.T.groupby(cell_indexer).mean().T

print("effects_by_cell shape:", effects_by_cell.shape)
display(effects_by_cell.iloc[:5, :5])

## 4. Dose–Response Analysis

**Goal.** Test whether Vitamin D analogs induce a monotonic transcriptomic response as dose increases, and quantify effect sizes (slopes).

**Method.**
1. Bin doses into "low" vs "high" categories for exploratory plots (`dose.binarize_dose`).
2. Test monotonicity with **Spearman correlation** (`dose.dose_monotonicity`).
3. Estimate slopes with **OLS regression** on log10(dose) (`dose.ols_hc3`) using HC3 robust errors.
4. Summarize slopes across cell lines and visualize with **forest plots** (`plotting.forest_from_models`).


### 4.1 Prepare dose metadata

In [ ]:
assert "dose" in META.columns, "Expected 'dose' in META after standardization."

META["log_dose"] = np.log10(META["dose"])
META["dose_bin"] = META.groupby("cell_id")["dose"].transform(lambda d: dose.binarize_dose(d).values)

META[["sig_id", "cell_id", "dose", "log_dose", "dose_bin"]].head()

In [ ]:
# 1) Build gene × cell effects
effects_by_cell = dataset.effects_by_cell(EXP, META)  # genes × cell_id
print("effects_by_cell:", effects_by_cell.shape)

# 2) Consensus core sets (UP/DOWN)
cons = coregenes.build_consensus_core(
    effects_by_cell,
    top_n=config.N_TOP,
    min_votes=config.VOTE_MIN,
    target_up=config.CORE_UP_N,
    target_dn=config.CORE_DN_N,
    min_non_na=10,
)
core_up_ids = cons["core_up"]
core_dn_ids = cons["core_dn"]
print(f"[core sets] UP={len(core_up_ids)} | DOWN={len(core_dn_ids)}")

# 3) Core score for every signature (columns of EXP)
core_scores = coregenes.core_score_for_matrix(
    effects=EXP,          # genes × sig_id
    core_up=core_up_ids,  # ID list (match EXP.index)
    core_dn=core_dn_ids,
    center=True,
)

# 4) Merge to META (standardized) by sig_id
if "core_score" in META.columns:
    META = META.drop(columns=["core_score"])
META = META.merge(core_scores.rename("core_score"),
                  left_on="sig_id", right_index=True, how="left")

# Sanity check
print("Has core_score?", "core_score" in META.columns, "| nulls:", META["core_score"].isna().sum())
display(META[["sig_id","cell_id","dose","core_score"]].head())


### 4.2 Monotonicity test (Spearman ρ)

In [ ]:
# Per-cell monotonicity of core_score vs dose
mono_results = (
    META.groupby("cell_id")
        .apply(lambda sub: dose.dose_monotonicity(sub["dose"], sub["core_score"]))
        .apply(pd.Series)
        .reset_index()
)

print(mono_results)